# M2 — Hierarchical Parsing and Standardization

Run this notebook from the code repository root in a CPU-only Colab session with the M1 corpus or checkpoint archives available. For the group's separate Jupyter Book repository, copy this notebook and the generated `data/tables/metrics_m2.json`; figures and tables are generated from that metrics file during book compilation. If the checkpoint run resumes, `last_run_wall_sec` measures only that invocation; `sum_batch_sec` includes compute time for every batch.

The parser starts at the `\documentclass` file, recursively follows active `\input` and `\include` commands, and ignores unused `.tex` files. It creates Document → heading → paragraph/list → sentence, formula, and figure leaves for every version. Reference sections are excluded; acknowledgements and appendices are retained.


## Standardization and deduplication

Formatting-only commands and float placement hints are removed. Inline math uses `$...$` and displays use `equation` or `equation*`; starred alignment uses `aligned` or `gathered` inside the display. Numbered multiline environments with per-row numbering or labels are retained to preserve semantics. `\bibitem` and cited entries from reachable `.bib` files are extracted; `\nocite{*}` includes the full library. Custom citation macros may require review. Compatible duplicate references are merged, their fields unioned, and citation keys rewritten. Elements with identical kind and cleaned content share an ID except where reusing an ancestor ID would form a cycle; hierarchy edges remain version-specific. Empty records and missing sources are counted separately.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

working = Path('data/working')
tables = Path('data/tables')
if not (tables / 'metrics_m2.json').exists():
    from src.m2_parsing.pipeline import run_milestone2
    from src.m2_parsing.batch_pipeline import run_checkpoint_batches
    if Path('range_1_dev').is_dir():
        run_checkpoint_batches(Path('range_1_dev'), working, tables, workers=2, resume=True)
    else:
        assert working.is_dir(), 'Place the M1 corpus in data/working first'
        run_milestone2(working_dir=working, tables_dir=tables)
metrics = json.loads((tables / 'metrics_m2.json').read_text(encoding='utf-8'))
coverage = pd.DataFrame(metrics['coverage'])
pd.DataFrame([{'publications': metrics['n_publications'], 'versions': metrics['n_versions_total'], 'parsed': metrics['n_versions_parsed'], 'empty': metrics['n_versions_empty'], 'failed': metrics['n_versions_failed'], 'success_rate': metrics['version_success_rate']}])


In [ ]:
coverage[['arxiv_id', 'n_versions', 'n_parsed', 'n_empty', 'n_elements_before_dedup', 'n_elements_after_dedup', 'n_bib_before', 'n_bib_after']]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].bar(['Before', 'After'], [metrics['n_elements_before_dedup'], metrics['n_elements_after_dedup']])
axes[0].set_title('Elements across versions')
axes[1].bar(['Before', 'After'], [metrics['n_bib_before'], metrics['n_bib_after']])
axes[1].set_title('BibTeX entries across versions')
plt.tight_layout()
plt.show()


In [ ]:
pd.DataFrame([{'last_run_wall_sec': metrics.get('end_to_end_sec'), 'reused_batches': metrics.get('reused_batches', 0), 'sum_batch_sec': metrics['duration_sec'], 'avg_pub_sec': metrics['avg_publication_sec'], 'last_run_peak_ram_mib': metrics['peak_rss_bytes'] / 2**20 if metrics['peak_rss_bytes'] else None, 'peak_worker_ram_mib': metrics.get('peak_worker_rss_bytes', 0) / 2**20, 'output_mib': metrics['output_bytes'] / 2**20}])
